# Matched perfect-correction channel and continuous-Lindblad analysis

This notebook reads one immutable completed run, verifies its metadata, and compares dephasing, domain-wall truncation, finite OW range, and dynamics family. Each plotting cell creates exactly one editable figure.

## Estimators and symmetry

The continuous generator has exact translation symmetry along $y$. The ordered random-schedule channel generally does not. Its displayed momentum spectrum is therefore computed only after the declared spatial sliding estimator: first average $G$ over cycles $N_y+1,\ldots,2N_y$, then apply the exact group twirl $N_y^{-1}\sum_s T_y^s G T_y^{-s}$. Untwirled translation residuals are always shown in the scalar table. Natural occupations are eigenvalues of $G$; entropy is labelled a Gaussian proxy whenever dephasing is present.

## Runtime and CPU allocation

In [ ]:
import os, sys
from pathlib import Path
CPU_RANGE = '0-7'  # edit before running
def parse_cpu_range(spec):
    cpus = set()
    for part in spec.split(','):
        bounds = [int(value) for value in part.strip().split('-')]
        cpus.update(range(bounds[0], bounds[-1] + 1))
    return cpus
if hasattr(os, 'sched_setaffinity'):
    os.sched_setaffinity(0, parse_cpu_range(CPU_RANGE))
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE))
import json, numpy as np, matplotlib.pyplot as plt
from analyze_campaign import load_cases, summarize, matched_pair_rows, validate_cross_arm_provenance, configure_plotting
configure_plotting()

## Load the immutable run and display actual parameters

In [ ]:
RUN_ROOT = Path(os.environ.get('MATCHED_RUN_ROOT', '/absolute/path/to/completed/run')).expanduser().resolve()  # set MATCHED_RUN_ROOT or edit
config = json.loads((RUN_ROOT / 'campaign_config.json').read_text())
manifest = json.loads((RUN_ROOT / 'manifest.json').read_text())
cases = load_cases(RUN_ROOT)
validate_cross_arm_provenance(cases)
rows = summarize(cases, float(config['analysis']['physicality_tolerance']))
pairs = matched_pair_rows(cases, rows)
display({
    'campaign': manifest['campaign'],
    'status': manifest['status'],
    'config_sha256': manifest['config_sha256'],
    'case_count': len(cases),
    'matched_pairs': len(pairs),
    'model': config['model'],
    'dynamics': config['dynamics'],
})

## Canonical occupation spectra

This plot compares the exact-twirl spectra at $N_y=64$, $\alpha_{\rm run,in}=1$, and DW-support truncation on.

In [ ]:
selected = [entry for entry in cases if entry['metadata']['case']['model']['Ny'] == 64 and entry['metadata']['case']['model']['alpha_run_in'] == 1.0 and entry['metadata']['case']['model']['dw_truncation'] and 'channel_schedule_seed_control' not in entry['metadata']['case']['campaign_roles']]
fig, axes = plt.subplots(1, 3, figsize=(7.05, 2.35), sharex=True, sharey=True)
arms = [('markov_channel', True), ('lindblad', False), ('lindblad', True)]
styles = {1: ('#D92725', '^'), 2: ('#2CA02C', 's'), None: ('#1F77B4', 'o')}
for ax, (family, deph) in zip(axes, arms):
    for entry in selected:
        case = entry['metadata']['case']; arrays = entry['arrays']
        if (case['dynamics']['family'], case['dynamics']['dephasing']) != (family, deph): continue
        nsh = case['model']['nshell']; color, marker = styles[nsh]
        occ = arrays['twirled_ky_natural_occupations'].mean(axis=0); ky = arrays['ky']
        ax.scatter(np.repeat(ky, occ.shape[1]), occ.ravel(), s=5, c=color, marker=marker, alpha=.55, label=f'nsh={nsh}', rasterized=True)
    ax.axhline(.5, color='.25', ls='--', lw=.7); ax.set_xlabel(r'$k_y$'); ax.set_title(f'{family}, deph={int(deph)}')
axes[0].set_ylabel(r'twirled occupation $\nu$'); axes[-1].legend(frameon=False)
fig.tight_layout(); plt.show()

## Four-estimator alpha scan

The global and bulk-excluded midpoint gaps, Gaussian entropy proxy, midgap wall weight, and relaxation gap are shown for every arm. The vertical line at $\alpha_{\rm run,in}=2$ is a parent-model reference, not a sampled point.

In [ ]:
scan = [row for row in rows if row['Ny'] == 64 and row['dw_truncation'] and row['sample_id'] == 0]
fig, axes = plt.subplots(2, 2, figsize=(7.05, 4.5), sharex=True)
arm_color = {('markov_channel', True): '#D92725', ('lindblad', False): '#2CA02C', ('lindblad', True): '#1F77B4'}
shell_marker = {1: '^', 2: 's', 'None': 'o'}
for (family, deph), color in arm_color.items():
    for nsh, marker in shell_marker.items():
        subset = sorted([row for row in scan if (row['dynamics'], row['dephasing'], row['nshell']) == (family, deph, nsh)], key=lambda row: row['alpha_run_in'])
        if not subset: continue
        alpha = [row['alpha_run_in'] for row in subset]; label = f'{family}, d={int(deph)}, nsh={nsh}'
        axes[0,0].semilogy(alpha, [row['twirled_half_occupation_gap'] for row in subset], color=color, marker=marker, label=label)
        axes[0,0].semilogy(alpha, [row['twirled_bulk_half_occupation_gap'] for row in subset], color=color, alpha=.35)
        axes[0,1].plot(alpha, [row['gaussian_entropy_proxy_per_circumference'] for row in subset], color=color, marker=marker)
        axes[1,0].plot(alpha, [row['midgap_wall_localization'] for row in subset], color=color, marker=marker)
        axes[1,1].semilogy(alpha, [row['relaxation_gap'] for row in subset], color=color, marker=marker)
for ax, ylabel in zip(axes.flat, [r'occupation gaps', r'$S_G/N_y$', 'midgap wall weight', 'relaxation gap']):
    ax.axvline(2, color='.25', ls='--', lw=.7); ax.set(xlabel=r'$\alpha_{\rm run,in}$', ylabel=ylabel)
axes[0,0].legend(frameon=False, fontsize=5, ncol=2)
fig.tight_layout(); plt.show()

## Schedule-origin and translation diagnostics

The ten-schedule control is the independent unit for channel schedule sensitivity.

In [ ]:
control = [row for row in rows if 'channel_schedule_seed_control' in row['campaign_roles']]
fig, ax = plt.subplots(figsize=(3.375, 2.45))
ax.scatter([row['sample_id'] for row in control], [row['late_translation_residual'] for row in control], color='#D92725', marker='^')
ax.set(xlabel='schedule sample', ylabel=r'$\|T_yGT_y^{-1}-G\|_F/\|G\|_F$')
fig.tight_layout(); plt.show()

## Density-kick response

For the ordered channel this is averaged over source positions only after retaining the source-resolved product. Response is computed for exactly 22 endpoint cases; alpha scans, $n_{\rm shell}=2$, and the $S=10$ schedule control have no response arrays.

In [ ]:
entry = next(entry for entry in cases if entry['metadata']['case']['model']['Ny'] == 64 and entry['metadata']['case']['model']['nshell'] == 1 and entry['metadata']['case']['model']['dw_truncation'] and entry['metadata']['case']['dynamics']['family'] == 'markov_channel' and 'channel_schedule_seed_control' not in entry['metadata']['case']['campaign_roles'])
response = entry['arrays']['response_density_ty_mean_source'].mean(axis=0); times = entry['arrays']['response_times']; vmax = np.max(np.abs(response))
fig, axes = plt.subplots(1, 2, figsize=(7.05, 2.35), sharex=True, sharey=True)
for wall, ax in enumerate(axes):
    im = ax.imshow(response[wall], aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax, extent=[0, 63, times[-1], times[0]])
    ax.set(xlabel=r'$y-y_0$', title=f'wall {wall + 1}')
axes[0].set_ylabel('cycle'); fig.colorbar(im, ax=axes, label=r'$\chi^{\rm wall}$')
fig.tight_layout(); plt.show()

## Cycle convergence

Cheap diagnostics are retained at every cycle; spectral quantities appear only at the five declared checkpoints.

In [ ]:
canonical = [entry for entry in cases if entry['metadata']['case']['model']['Ny'] == 64 and entry['metadata']['case']['model']['alpha_run_in'] == 1.0 and entry['metadata']['case']['model']['nshell'] == 1 and entry['metadata']['case']['model']['dw_truncation'] and 'channel_schedule_seed_control' not in entry['metadata']['case']['campaign_roles']]
fig, axes = plt.subplots(1, 2, figsize=(7.05, 2.45))
for entry, color in zip(sorted(canonical, key=lambda e: (e['metadata']['case']['dynamics']['family'] != 'markov_channel', e['metadata']['case']['dynamics']['dephasing'])), ['#D92725','#2CA02C','#1F77B4']):
    case = entry['metadata']['case']; arrays = entry['arrays']; label = f"{case['dynamics']['family']}, d={int(case['dynamics']['dephasing'])}"
    axes[0].semilogy(arrays['cycle']/64, np.maximum(arrays['successive_state_distance'].mean(axis=0), 1e-18), color=color, label=label)
    axes[1].semilogy(arrays['spectral_checkpoint_cycle']/64, np.maximum(arrays['half_occupation_gap'].mean(axis=0), 1e-18), color=color, marker='o')
axes[0].set(xlabel=r'$c/N_y$', ylabel='successive-state distance'); axes[0].legend(frameon=False)
axes[1].set(xlabel=r'$c/N_y$', ylabel=r'$\min|\nu-1/2|$')
fig.tight_layout(); plt.show()

## Response velocity versus size

Response exists only for the declared 22 endpoint cases: finite $n_{\rm shell}=1$ for all three arms and sizes, plus full-frame continuous controls at $N_y=64$.

In [ ]:
response_rows = [row for row in rows if row['response_source_count'] > 0 and row['alpha_run_in'] == 1 and row['sample_id'] == 0]
fig, axes = plt.subplots(1, 2, figsize=(7.05, 2.45), sharex=True, sharey=True)
for wall, ax in enumerate(axes, start=1):
    for (family, deph), color in [(('markov_channel',True),'#D92725'), (('lindblad',False),'#2CA02C'), (('lindblad',True),'#1F77B4')]:
        for dw, ls in [(True,'-'), (False,'--')]:
            subset = sorted([row for row in response_rows if (row['dynamics'],row['dephasing'],row['nshell'],row['dw_truncation']) == (family,deph,1,dw)], key=lambda row: row['Ny'])
            if subset: ax.plot([1/row['Ny'] for row in subset], [row[f'response_velocity_wall_{wall}_mean_source'] for row in subset], color=color, marker='^', ls=ls, label=f'{family}, d={int(deph)}, dw={int(dw)}')
    ax.axhline(0, color='.25', lw=.7); ax.set(xlabel=r'$1/N_y$', title=f'wall {wall}')
axes[0].set_ylabel('positive-lobe velocity'); axes[1].legend(frameon=False, fontsize=5)
fig.tight_layout(); plt.show()

## Direct channel--continuous comparison

These are finite unit-step differences, not a weak-step convergence fit. Each pair has identical OW projector hashes and model parameters.

In [ ]:
endpoint_pairs = [row for row in pairs if row['alpha_run_in'] == 1]
fig, axes = plt.subplots(1, 2, figsize=(7.05, 2.45), sharex=True)
for nsh, color, marker in [(1,'#D92725','^'), (2,'#2CA02C','s')]:
    for dw, ls in [(True,'-'), (False,'--')]:
        subset = sorted([row for row in endpoint_pairs if row['nshell'] == nsh and row['dw_truncation'] == dw], key=lambda row: row['Ny'])
        if not subset: continue
        x = [1/row['Ny'] for row in subset]; label = f'nsh={nsh}, dw={int(dw)}'
        axes[0].semilogy(x, [row['state_relative_frobenius_error'] for row in subset], color=color, marker=marker, ls=ls, label=label)
        axes[1].semilogy(x, [row['occupation_rms_difference'] for row in subset], color=color, marker=marker, ls=ls)
axes[0].set(xlabel=r'$1/N_y$', ylabel='state relative error'); axes[0].legend(frameon=False)
axes[1].set(xlabel=r'$1/N_y$', ylabel='occupation RMS difference')
fig.tight_layout(); plt.show()

## Raw diagnostics

In [ ]:
print('rows:', len(rows))
print('nonphysical:', sum(not row['physical'] for row in rows))
print('matched channel--continuous pairs:', len(pairs))
print('maximum Lindblad translation residual:', max(row['late_translation_residual'] for row in rows if row['dynamics'] == 'lindblad'))
display(rows[:5])
display(pairs[:5])